# Robust Optimization

Real-world optimization problems rarely have perfectly known parameters. Demand forecasts carry error, material costs fluctuate, and physical measurements are imprecise. **Robust optimization** provides a principled framework for making decisions that remain feasible — and near-optimal — for *every* realization of uncertain parameters within a prescribed set {cite:p}`BenTal1999,BenTal2009`.

## What is Robust Optimization?

Given a nominal problem

$$
\min_x\; f(x, p) \quad \text{s.t.}\; g_i(x, p) \le 0,\; i=1,\ldots,m
$$

where $p$ is an uncertain parameter taking values in an **uncertainty set** $\mathcal{U}$, the robust counterpart is:

$$
\min_x\; \max_{p \in \mathcal{U}}\; f(x, p) \quad \text{s.t.}\; g_i(x, p) \le 0\; \forall p \in \mathcal{U},\; i=1,\ldots,m
$$

A key insight (Ben-Tal & Nemirovski {cite:t}`BenTal1999`) is that for standard uncertainty set shapes, this minimax problem can be *reformulated* as a deterministic program without nested optimization.

## Uncertainty Sets

discopt supports three standard uncertainty set families:

| Set | Definition | Reformulation |
|---|---|---|
| **Box** | $\lvert p_j - \bar{p}_j \rvert \le \delta_j$ | Substitute worst-case bound per component |
| **Ellipsoidal** | $\lVert \Sigma^{-1/2}(p-\bar{p}) \rVert_2 \le \rho$ | Add SOC norm penalty {cite:p}`BenTal1999` |
| **Polyhedral** | $A\xi \le b,\; \xi = p - \bar{p}$ | LP dual auxiliary variables {cite:p}`BertsimasSim2004` |

In [ ]:
import discopt.modeling as dm
import numpy as np
from discopt.ro import (
    BoxUncertaintySet,
    EllipsoidalUncertaintySet,
    RobustCounterpart,
    budget_uncertainty_set,
)

## Example 1 — Production Planning with Box Uncertainty

A manufacturer produces three products. Unit costs and demand are uncertain:

$$
\min_x\; c^\top x \quad
\text{s.t.}\; \mathbf{1}^\top x \ge d,\; x \ge 0
$$

where $c \in [\bar{c} - \delta_c, \bar{c} + \delta_c]$ and $d \in [\bar{d} - \delta_d, \bar{d} + \delta_d]$.

The robust counterpart simply tightens the cost coefficients and demand bound to their worst-case values.

In [ ]:
# Nominal model
m = dm.Model("production")
x = m.continuous("x", shape=(3,), lb=0)

c_bar = np.array([10.0, 15.0, 8.0])  # nominal unit costs
d_bar = 100.0  # nominal demand

c = m.parameter("c", value=c_bar)
d = m.parameter("d", value=d_bar)

m.minimize(dm.sum(c * x))
m.subject_to(dm.sum(x) >= d, name="demand")

print("Nominal objective:", m._objective)
print("Constraint:       ", m._constraints[0])

In [ ]:
# Declare uncertainty: costs ± 10 %, demand ± 5 %
unc_c = BoxUncertaintySet(c, delta=0.10 * c_bar)
unc_d = BoxUncertaintySet(d, delta=0.05 * d_bar)

print(f"Cost bounds:   {unc_c.lower} to {unc_c.upper}")
print(f"Demand bounds: {unc_d.lower[()]} to {unc_d.upper[()]}")

In [ ]:
# Build robust counterpart (modifies m in-place)
rc = RobustCounterpart(m, [unc_c, unc_d])
rc.formulate()

print("Robust objective:", m._objective)
print("Robust constraint:", m._constraints[0])

After calling `formulate()`, the uncertain parameters `c` and `d` are replaced by their worst-case values. The robust model is now a *deterministic* LP with tightened cost coefficients `[11.0, 16.5, 8.8]` and a tightened demand `105.0`.

## Example 2 — Portfolio Optimization with Ellipsoidal Uncertainty

Portfolio optimization with uncertain expected returns {cite:p}`BenTal1999`:

$$
\max_x\; \mu^\top x \quad
\text{s.t.}\; x^\top \Sigma x \le \sigma^2_{\max},\;
\mathbf{1}^\top x = 1,\; x \ge 0
$$

where returns $\mu$ lie in an ellipsoidal uncertainty set $\{\mu: \lVert \mu - \bar{\mu} \rVert_2 \le \rho\}$.

The robust counterpart of the objective becomes:

$$
\max_x\; \bar{\mu}^\top x - \rho \lVert x \rVert_2
$$

In [ ]:
rng = np.random.RandomState(42)
n_assets = 5

mu_bar = rng.uniform(0.05, 0.15, n_assets)  # expected returns
sigma_max = 0.20  # max portfolio volatility

m_port = dm.Model("portfolio")
x_port = m_port.continuous("x", shape=(n_assets,), lb=0, ub=1)
mu = m_port.parameter("mu", value=mu_bar)

# Maximize expected return (minimize negative return)
m_port.minimize(-(mu @ x_port))
m_port.subject_to(dm.sum(x_port) == 1.0, name="budget")

# Ellipsoidal uncertainty on returns: rho=0.02 (±2% shift in any direction)
unc_mu = EllipsoidalUncertaintySet(mu, rho=0.02)

rc_port = RobustCounterpart(m_port, unc_mu)
rc_port.formulate()

print("Robust portfolio objective includes penalty term (sqrt node):")


def _show_tree(expr, indent=0):
    from discopt.modeling.core import (
        BinaryOp,
        Constant,
        FunctionCall,
        MatMulExpression,
        Parameter,
        UnaryOp,
    )

    prefix = "  " * indent
    if isinstance(expr, Constant):
        print(f"{prefix}Constant({np.round(expr.value, 3)})")
    elif isinstance(expr, Parameter):
        print(f"{prefix}Param({expr.name})")
    elif isinstance(expr, BinaryOp):
        print(f"{prefix}BinaryOp({expr.op})")
        _show_tree(expr.left, indent + 1)
        _show_tree(expr.right, indent + 1)
    elif isinstance(expr, UnaryOp):
        print(f"{prefix}UnaryOp({expr.op})")
        _show_tree(expr.operand, indent + 1)
    elif isinstance(expr, MatMulExpression):
        print(f"{prefix}MatMul")
        _show_tree(expr.left, indent + 1)
        _show_tree(expr.right, indent + 1)
    elif isinstance(expr, FunctionCall):
        print(f"{prefix}FunctionCall({expr.func_name})")
        for a in expr.args:
            _show_tree(a, indent + 1)
    else:
        print(f"{prefix}{type(expr).__name__}")


_show_tree(m_port._objective.expression)

## Example 3 — Budget of Uncertainty (Bertsimas & Sim)

The **budget-of-uncertainty** set {cite:p}`BertsimasSim2004` limits the number of simultaneously adversarial parameters:

$$
\mathcal{U}_\Gamma = \left\{ p : |p_j - \bar{p}_j| \le \delta_j,\;
\sum_j \frac{|p_j - \bar{p}_j|}{\delta_j} \le \Gamma \right\}
$$

When $\Gamma = 0$: deterministic (nominal solution only). When $\Gamma = k$: full box uncertainty.

The budget $\Gamma$ gives a direct trade-off between **solution cost** and **protection level**.

In [ ]:
n = 5
c_nom = np.array([10.0, 12.0, 8.0, 15.0, 9.0])
delta = 0.1 * c_nom


def build_budget_model(gamma):
    """Build a robust production model with budget of uncertainty Gamma."""
    m = dm.Model(f"budget_gamma={gamma}")
    x = m.continuous("x", shape=(n,), lb=0)
    c = m.parameter("c", value=c_nom)
    m.minimize(dm.sum(c * x))
    m.subject_to(dm.sum(x) >= 50.0, name="demand")
    unc = budget_uncertainty_set(c, delta=delta, gamma=gamma)
    rc = RobustCounterpart(m, unc)
    rc.formulate()
    return m


def _get_constants(expr):
    from discopt.modeling.core import (
        BinaryOp,
        Constant,
        FunctionCall,
        MatMulExpression,
        SumExpression,
        UnaryOp,
    )

    if isinstance(expr, Constant):
        return [float(np.sum(expr.value))]
    if isinstance(expr, (BinaryOp, MatMulExpression)):
        return _get_constants(expr.left) + _get_constants(expr.right)
    if isinstance(expr, UnaryOp):
        return _get_constants(expr.operand)
    if isinstance(expr, SumExpression):
        return _get_constants(expr.operand)
    if isinstance(expr, FunctionCall):
        r = []
        for a in expr.args:
            r += _get_constants(a)
        return r
    return []


for gamma in [0, 1, 2, 3, 5]:
    m_g = build_budget_model(gamma)
    vals = _get_constants(m_g._objective.expression)
    print(f"Γ={gamma}: worst-case cost components (sum) = {sum(vals):.2f}")

As $\Gamma$ increases, more uncertainty is accounted for, leading to a higher worst-case cost estimate. At $\Gamma = 0$ the model reduces to the nominal problem; at $\Gamma = k = 5$ every component is at its worst-case value simultaneously.

## Multiple Uncertain Parameters

Multiple parameters from the same uncertainty family can be passed as a list:

In [ ]:
m_multi = dm.Model("multi_uncertain")
x = m_multi.continuous("x", shape=(3,), lb=0)
c = m_multi.parameter("c", value=[10.0, 15.0, 8.0])
d = m_multi.parameter("d", value=50.0)

m_multi.minimize(dm.sum(c * x))
m_multi.subject_to(dm.sum(x) >= d, name="demand")
m_multi.subject_to(x[0] + 2 * x[1] <= 80.0, name="resource")

rc_multi = RobustCounterpart(
    m_multi,
    [
        BoxUncertaintySet(c, delta=[1.0, 1.5, 0.8]),
        BoxUncertaintySet(d, delta=5.0),
    ],
)
rc_multi.formulate()

print("Robust constraints:")
for con in m_multi._constraints:
    print(f"  [{con.name}]  {con.body} {con.sense} {con.rhs}")

## API Summary

```python
from discopt.ro import (
    AffineDecisionRule,        # adjustable robust (affine policy)
    BoxUncertaintySet,         # per-component interval
    EllipsoidalUncertaintySet, # ellipsoidal ball
    PolyhedralUncertaintySet,  # general polytope
    RobustCounterpart,         # static robust counterpart
    budget_uncertainty_set,    # Bertsimas-Sim convenience constructor
)

# -- Static robust optimization -----------------------------------------------
m = dm.Model()
p = m.parameter("p", value=nominal_value)
# ... build objective and constraints using p ...
rc = RobustCounterpart(m, BoxUncertaintySet(p, delta=0.1 * nominal_value))
rc.formulate()          # modifies m in-place; p replaced by worst-case constant
result = m.solve()

# -- Adjustable robust optimization (two-stage) -------------------------------
m = dm.Model()
x = m.continuous("x", ...)    # here-and-now decisions
y = m.continuous("y", ...)    # wait-and-see recourse
p = m.parameter("p", value=nominal_value)
# ... build objective and constraints using x, y, p ...

# Stage 1: affine rule  y(xi) = y0 + Y0*(p - p_bar)
adr = AffineDecisionRule(y, uncertain_params=p)
adr.apply()                    # retires y; adds y0, Y0 as decision variables

# Stage 2: handle remaining xi terms
rc = RobustCounterpart(m, BoxUncertaintySet(p, delta=delta))
rc.formulate()                 # fully deterministic model

result = m.solve()             # optimises over (x, y0, Y0)
```

## Limitations

- **Coefficient uncertainty** (bilinear `p * x`) requires LP dual auxiliary variables; planned for a future release.
- **Recourse shape**: `AffineDecisionRule` supports scalar and 1-D recourse variables; 2-D support is planned.
- **Non-affine recourse** (polynomial, piecewise-linear) is not supported.
- **Distributionally robust optimization** (moment-based or Wasserstein ambiguity sets) is not yet supported.

For a full adjustable RO tutorial including multi-parameter recourse and vector recourse variables,
see {doc}`robust_adjustable`.

## References

```{bibliography}
:filter: docname in docnames
```